In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [28]:
import os, re, math, time, random, pickle, hashlib
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, FrozenSet

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

import rasterio
from tqdm.auto import tqdm

In [29]:

# ----------------------------
# Task config
# ----------------------------
NUM_CLASSES = 4
ACTIVITY_NAMES = ["no_activity", "plowing", "harvesting", "burning"]  # index = class id

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [30]:

# Scan filenames (same naming convention, cl now in {0,1,2,3})
# im_id1_cl_id2_date1_date2_area.tif  (id1 territory, cl label, id2 field)

FNAME_RE_OLD = re.compile(
    r"^im_(?P<id1>-?\d+)_(?P<cl>[0-3])_(?P<id2>-?\d+)_(?P<dmg>\d{8})_(?P<ref>\d{8})_(?P<area>[-+]?\d*\.?\d+)\.tif$"
)

@dataclass(frozen=True)
class SampleMeta:
    path: str
    id1: int
    id2: int
    cl: int          # 0=no_activity, 1=plowing, 2=harvesting, 3=burning
    dmg: str
    ref: str
    area: float
    field_key: Tuple[int, int]  # (id1, id2)


In [31]:

def scan_folder(folder: str) -> List[SampleMeta]:
    out = []
    for fn in os.listdir(folder):
        if not fn.lower().endswith(".tif"):
            continue
        m = FNAME_RE_OLD.match(fn)
        if not m:
            continue
        id1 = int(m.group("id1"))
        id2 = int(m.group("id2"))
        cl = int(m.group("cl"))
        if cl not in range(NUM_CLASSES):
            raise ValueError(f"Unexpected class {cl} parsed from filename {fn}")
        out.append(SampleMeta(
            path=os.path.join(folder, fn),
            id1=id1,
            id2=id2,
            cl=cl,
            dmg=m.group("dmg"),
            ref=m.group("ref"),
            area=float(m.group("area")),
            field_key=(id1, id2)
        ))
    if not out:
        raise RuntimeError(f"No matching tif found in {folder}")
    return out

In [32]:

# ----------------------------
# Field split with 20% val per group
# Groups are defined by the SET of activity classes seen at a field
# ( grouping to 4 classes)
# ----------------------------
def make_field_split(samples: List[SampleMeta], val_frac_each_group=0.20, seed=42):
    rng = random.Random(seed)
    field_to_idxs: Dict[Tuple[int, int], List[int]] = {}
    for i, s in enumerate(samples):
        field_to_idxs.setdefault(s.field_key, []).append(i)

    # signature = which classes ever occur at this field (usually a single class,
    # but a field can have multiple date-pairs with different outcomes)
    field_signature: Dict[Tuple[int, int], FrozenSet[int]] = {
        fk: frozenset(samples[j].cl for j in idxs)
        for fk, idxs in field_to_idxs.items()
    }

    groups: Dict[FrozenSet[int], List[Tuple[int, int]]] = {}
    for fk, sig in field_signature.items():
        groups.setdefault(sig, []).append(fk)

    val_fields = set()
    for sig, fields in groups.items():
        fields = fields[:]
        rng.shuffle(fields)
        n_val = int(round(val_frac_each_group * len(fields)))
        n_val = max(1, n_val)
        if len(fields) > 1:
            # never drain a whole group into val - keep at least 1 field for training
            n_val = min(n_val, len(fields) - 1)
        else:
            n_val = 0  # a single-field group stays in train; can't split it safely
        val_fields.update(fields[:n_val])

    train_fields = set(field_to_idxs.keys()) - val_fields

    train_idx = [i for fk in train_fields for i in field_to_idxs[fk]]
    val_idx   = [i for fk in val_fields   for i in field_to_idxs[fk]]
    return train_idx, val_idx


In [33]:

# Dataset: same preprocessing as before (mask from band1 & band6 intersection,
# robust med/MAD norm on valid, remask). Only the label dtype changes:
# long class index (multiclass).
# ----------------------------
class TifDatasetLikeTraining(Dataset):
    def __init__(self, samples: List[SampleMeta], indices: List[int], normalize=True):
        self.samples = samples
        self.indices = indices
        self.normalize = normalize
        self.eps = 1e-6
        self.b1 = 0
        self.b6 = 5

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        meta = self.samples[self.indices[k]]
        with rasterio.open(meta.path) as src:
            arr = src.read().astype(np.float32)  # [10,H,W]
        x = torch.from_numpy(arr)

        b1 = x[self.b1]; b6 = x[self.b6]
        valid = torch.isfinite(b1) & torch.isfinite(b6) & (b1 != 0) & (b6 != 0)

        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        if self.normalize and valid.any():
            for b in range(x.shape[0]):
                vals = x[b][valid]
                if vals.numel() < 10:
                    continue
                med = vals.median()
                mad = (vals - med).abs().median()
                scale = 1.4826 * mad
                if (not torch.isfinite(scale)) or (scale < self.eps):
                    continue
                x[b] = (x[b] - med) / (scale + self.eps)

        x[:, ~valid] = 0.0
        y = torch.tensor(meta.cl, dtype=torch.long)  # class index
        return x, y, valid


def collate_pad(batch):
    xs, ys, vms = zip(*batch)
    B = len(xs); C = xs[0].shape[0]
    Hmax = max(x.shape[1] for x in xs)
    Wmax = max(x.shape[2] for x in xs)

    x_pad = torch.zeros((B, C, Hmax, Wmax), dtype=torch.float32)
    vm_pad = torch.zeros((B, Hmax, Wmax), dtype=torch.bool)
    y = torch.stack(list(ys), dim=0)  # [B], long

    for b in range(B):
        x = xs[b]; vm = vms[b]
        H, W = x.shape[1], x.shape[2]
        x_pad[b, :, :H, :W] = x
        vm_pad[b, :H, :W] = vm

    return x_pad, y, vm_pad


In [34]:
# ----------------------------
# Balanced batch sampler, generalized from 50/50 to 1/4 per class.
# ----------------------------
class BalancedBatchSampler(Sampler[List[int]]):
    def __init__(self, dataset: Dataset, batch_size: int, num_classes: int = NUM_CLASSES, seed: int = 0):
        assert batch_size % num_classes == 0, "batch_size must be divisible by num_classes"
        self.dataset = dataset
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.per_class = batch_size // num_classes
        self.rng = random.Random(seed)

        buckets: Dict[int, List[int]] = {c: [] for c in range(num_classes)}
        for di, global_i in enumerate(dataset.indices):
            cl = dataset.samples[global_i].cl
            buckets[cl].append(di)

        missing = [c for c, idxs in buckets.items() if not idxs]
        if missing:
            names = [ACTIVITY_NAMES[c] for c in missing]
            raise RuntimeError(f"Need all {num_classes} classes present. Missing: {names}")

        self.buckets = buckets

    def __iter__(self):
        n_batches = math.ceil(max(len(v) for v in self.buckets.values()) / self.per_class)
        pools = {c: idxs[:] for c, idxs in self.buckets.items()}
        for pool in pools.values():
            self.rng.shuffle(pool)

        def take(pool, src, k):
            out = []
            while len(out) < k:
                if not pool:
                    pool.extend(src[:])
                    self.rng.shuffle(pool)
                out.append(pool.pop())
            return out

        for _ in range(n_batches):
            batch = []
            for c in range(self.num_classes):
                batch += take(pools[c], self.buckets[c], self.per_class)
            self.rng.shuffle(batch)
            yield batch

    def __len__(self):
        return math.ceil(max(len(v) for v in self.buckets.values()) / self.per_class)

In [35]:
# ----------------------------
# Metrics & evaluation for multiclass
# (argmax prediction, per-class + macro metrics)
# ----------------------------
def confusion_matrix_multiclass(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = NUM_CLASSES) -> np.ndarray:
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true.astype(int), y_pred.astype(int)):
        cm[t, p] += 1
    return cm

def multiclass_metrics(cm: np.ndarray) -> dict:
    num_classes = cm.shape[0]
    n = int(cm.sum())
    per_class = {}
    f1s = []
    for c in range(num_classes):
        tp = int(cm[c, c])
        fp = int(cm[:, c].sum() - tp)
        fn = int(cm[c, :].sum() - tp)
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        f1 = 2 * precision * recall / max(1e-12, precision + recall)
        per_class[c] = dict(precision=precision, recall=recall, f1=f1, support=int(cm[c, :].sum()))
        f1s.append(f1)
    overall_acc = float(np.trace(cm)) / max(1, n)
    macro_f1 = float(np.mean(f1s))
    return dict(per_class=per_class, overall_accuracy=overall_acc, macro_f1=macro_f1, N=n)

@torch.no_grad()
def predict_logits(model, loader, device="cuda", use_amp=True):
    model.eval()
    logits_all, y_all = [], []
    for x, y, vm in tqdm(loader, desc="VAL inference", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        vm = vm.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(use_amp and device == "cuda")):
            logits = model(x, valid_mask=vm)  # [B, NUM_CLASSES]
        logits_all.append(logits.float().cpu().numpy())
        y_all.append(y.cpu().numpy())
    return np.concatenate(logits_all), np.concatenate(y_all)

def full_validation_report(model, val_loader, device="cuda", num_classes=NUM_CLASSES, use_amp=True) -> dict:
    logits, y = predict_logits(model, val_loader, device=device, use_amp=use_amp)
    y_pred = logits.argmax(axis=1)
    cm = confusion_matrix_multiclass(y, y_pred, num_classes=num_classes)
    report = multiclass_metrics(cm)
    report["confusion_matrix"] = cm
    return report

def print_report(report: dict):
    print("=== FULL VAL REPORT ===")
    print(f"overall_accuracy={report['overall_accuracy']:.3f}  macro_f1={report['macro_f1']:.3f}  N={report['N']}")
    for c, m in report["per_class"].items():
        print(f"  [{c}:{ACTIVITY_NAMES[c]:<11}] precision={m['precision']:.3f} recall={m['recall']:.3f} "
              f"f1={m['f1']:.3f} support={m['support']}")
    print("  confusion matrix (rows=true, cols=pred):")
    print(report["confusion_matrix"])

In [36]:
# ----------------------------
# Shared training loop - CrossEntropyLoss
# argmax accuracy
# ----------------------------
def train_model(
    model: nn.Module,
    train_loader,
    val_loader,
    device=None,
    epochs=5,
    lr=3e-4,
    weight_decay=1e-4,
    use_amp=True,
    val_every=50,
    log_every=10,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[4, 6, 8], gamma=0.2)
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))
    criterion = nn.CrossEntropyLoss()

    global_step = 0
    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_corr = 0
        running_n = 0
        t0 = time.time()

        pbar = tqdm(enumerate(train_loader, start=1), total=len(train_loader), desc=f"TRAIN epoch {ep}/{epochs}")
        val_iter = iter(val_loader)

        for bi, (x, y, vm) in pbar:
            global_step += 1
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)  # long [B]
            vm = vm.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(use_amp and device == "cuda")):
                logits = model(x, valid_mask=vm)  # [B, NUM_CLASSES]
                loss = criterion(logits, y)

            if not torch.isfinite(loss):
                print(f" non-finite loss at step={global_step}, skipping")
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            pred = logits.detach().argmax(dim=1)
            corr = int((pred == y).sum().item())
            n = int(y.numel())

            running_loss += float(loss.item()) * x.size(0)
            running_corr += corr
            running_n += n

            avg_loss = running_loss / max(1, running_n)
            avg_acc = running_corr / max(1, running_n)
            lr_now = opt.param_groups[0]["lr"]

            if (global_step % val_every) == 0:
                try:
                    vx, vy, vvm = next(val_iter)
                except StopIteration:
                    val_iter = iter(val_loader)
                    vx, vy, vvm = next(val_iter)
                vx = vx.to(device); vy = vy.to(device); vvm = vvm.to(device)
                with torch.amp.autocast("cuda", enabled=(use_amp and device == "cuda")):
                    vlogits = model(vx, valid_mask=vvm)
                    vloss = criterion(vlogits, vy).item()
                    vacc = (vlogits.argmax(dim=1) == vy).float().mean().item()
                pbar.set_postfix({"lr": f"{lr_now:.2e}", "loss": f"{loss.item():.4f}", "acc": f"{corr/n:.3f}",
                                  "avg_loss": f"{avg_loss:.4f}", "avg_acc": f"{avg_acc:.3f}",
                                  "val_b_loss": f"{vloss:.4f}", "val_b_acc": f"{vacc:.3f}"})
            else:
                pbar.set_postfix({"lr": f"{lr_now:.2e}", "loss": f"{loss.item():.4f}", "acc": f"{corr/n:.3f}",
                                  "avg_loss": f"{avg_loss:.4f}", "avg_acc": f"{avg_acc:.3f}"})

            if (bi % log_every) == 0:
                print(f"[ep{ep} b{bi:04d} step{global_step}] loss={loss.item():.4f} avg_loss={avg_loss:.4f} "
                      f"avg_acc={avg_acc:.3f} lr={lr_now:.2e} elapsed={time.time()-t0:.1f}s")

        sched.step()
        print(f"\n=== Epoch {ep} complete | TRAIN avg_loss={avg_loss:.4f} avg_acc={avg_acc:.4f} | "
              f"lr now {opt.param_groups[0]['lr']:.2e}")

        report = full_validation_report(model, val_loader, device=device, num_classes=NUM_CLASSES, use_amp=use_amp)
        print_report(report)

    return model

In [42]:
# ----------------------------
# Data loading
# ----------------------------
DATA_FOLDER = r"/content/tr_data_cutted"

samples = scan_folder(DATA_FOLDER)
train_idx, val_idx = make_field_split(samples, val_frac_each_group=0.20, seed=42)

train_ds = TifDatasetLikeTraining(samples, train_idx, normalize=True)
val_ds   = TifDatasetLikeTraining(samples, val_idx,   normalize=True)

BATCH_SIZE = 8   #divisible by NUM_CLASSES (4) -> 2 samples/class/batch
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds,
    batch_sampler=BalancedBatchSampler(train_ds, batch_size=BATCH_SIZE, num_classes=NUM_CLASSES, seed=42),
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_pad,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_pad,
)

print("Loaders ready  (4-class balanced sampling)")

# ----------------------------
# Split / class-balance diagnostics
# ----------------------------
from collections import Counter

def count_distinct_events(samples: List[SampleMeta], indices: List[int]) -> Dict[int, int]:
    """Distinct (field, end-date) pairs per class - ignores how many start-dates
    were paired with each one, so this reflects real diversity, not augmented count."""
    events_by_class: Dict[int, set] = {c: set() for c in range(NUM_CLASSES)}
    for i in indices:
        m = samples[i]
        events_by_class[m.cl].add((m.id1, m.id2, m.dmg))
    return {c: len(v) for c, v in events_by_class.items()}

train_field_count = len(set(samples[i].field_key for i in train_idx))
val_field_count   = len(set(samples[i].field_key for i in val_idx))
print(f"\nTrain: {len(train_idx)} samples across {train_field_count} fields")
print(f"Val:   {len(val_idx)} samples across {val_field_count} fields")

train_counts = Counter(samples[i].cl for i in train_idx)
val_counts   = Counter(samples[i].cl for i in val_idx)
train_events = count_distinct_events(samples, train_idx)
val_events   = count_distinct_events(samples, val_idx)

print("\nPer-class breakdown (samples = rows incl. start-date pairings, events = distinct real occurrences):")
for c in range(NUM_CLASSES):
    print(f"  [{c}:{ACTIVITY_NAMES[c]:<11}] "
          f"train: samples={train_counts[c]:4d} events={train_events[c]:4d}  |  "
          f"val: samples={val_counts[c]:4d} events={val_events[c]:4d}")

Loaders ready  (4-class balanced sampling)

Train: 13906 samples across 774 fields
Val:   3460 samples across 194 fields

Per-class breakdown (samples = rows incl. start-date pairings, events = distinct real occurrences):
  [0:no_activity] train: samples=13221 events=12594  |  val: samples=3298 events=3143
  [1:plowing    ] train: samples= 346 events=  59  |  val: samples=  94 events=  16
  [2:harvesting ] train: samples= 303 events=  80  |  val: samples=  62 events=  19
  [3:burning    ] train: samples=  36 events=   6  |  val: samples=   6 events=   1


In [ ]:
# ----------------------------
# Model: FastCNNClassifier, output head now has NUM_CLASSES logits
# ----------------------------
def resize_mask_like(mask: torch.Tensor, feat: torch.Tensor) -> torch.Tensor:
    if mask is None:
        return None
    Hf, Wf = feat.shape[-2], feat.shape[-1]
    m = mask.unsqueeze(1).float()
    m = F.interpolate(m, size=(Hf, Wf), mode="nearest")
    return (m.squeeze(1) > 0)

def safe_maxpool2d(x: torch.Tensor, k: int = 2, s: int = 2) -> torch.Tensor:
    H, W = x.shape[-2], x.shape[-1]
    if H < k or W < k:
        return x
    return F.max_pool2d(x, kernel_size=k, stride=s)

def safe_pool_mask(mask: torch.Tensor, x_after_pool: torch.Tensor) -> torch.Tensor:
    return resize_mask_like(mask, x_after_pool)

def masked_global_avg_pool(feat: torch.Tensor, mask: torch.Tensor):
    m = mask.unsqueeze(1).float()
    denom = m.sum(dim=(2, 3)).clamp(min=1.0)
    return (feat * m).sum(dim=(2, 3)) / denom


class FastCNNClassifier(nn.Module):
    def __init__(self, in_ch=10, base=64, num_classes=NUM_CLASSES):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, padding=1),
            nn.BatchNorm2d(base),
            nn.GELU(),
            nn.Conv2d(base, base, 3, padding=1),
            nn.BatchNorm2d(base),
            nn.GELU(),
        )
        self.b2 = nn.Sequential(
            nn.Conv2d(base, base*2, 3, padding=1),
            nn.BatchNorm2d(base*2),
            nn.GELU(),
            nn.Conv2d(base*2, base*2, 3, padding=1),
            nn.BatchNorm2d(base*2),
            nn.GELU(),
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(base*2, base*4, 3, padding=1),
            nn.BatchNorm2d(base*4),
            nn.GELU(),
            nn.Conv2d(base*4, base*4, 3, padding=1),
            nn.BatchNorm2d(base*4),
            nn.GELU(),
        )
        self.head = nn.Sequential(
            nn.Linear(base*4, base*2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(base*2, num_classes)
        )

    def forward(self, x, valid_mask=None):
        m = valid_mask  # [B,H,W]

        x = self.b1(x)
        m = resize_mask_like(m, x)

        x = safe_maxpool2d(x, 2, 2)
        m = safe_pool_mask(m, x)

        x = self.b2(x)
        m = resize_mask_like(m, x)

        x = safe_maxpool2d(x, 2, 2)
        m = safe_pool_mask(m, x)

        x = self.b3(x)
        m = resize_mask_like(m, x)

        pooled = masked_global_avg_pool(x, m) if m is not None else x.mean(dim=(2, 3))
        return self.head(pooled)  # [B, NUM_CLASSES] logits - no squeeze


cnn = FastCNNClassifier(in_ch=10, base=64, num_classes=NUM_CLASSES)
cnn = train_model(cnn, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-4,
                   val_every=10000, log_every=10000)
SAVE_PATH = os.path.abspath("cnn_4class_final_10epoch.pth")
torch.save(cnn.state_dict(), SAVE_PATH)
print(f"Model weights saved to: {SAVE_PATH}")

/tmp/ipykernel_834/346543900.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))


TRAIN epoch 1/10:   0%|          | 0/6611 [00:00<?, ?it/s]


=== Epoch 1 complete | TRAIN avg_loss=0.2236 avg_acc=0.9338 | lr now 1.00e-03


VAL inference:   0%|          | 0/433 [00:00<?, ?it/s]

=== FULL VAL REPORT ===
overall_accuracy=0.916  macro_f1=0.414  N=3460
  [0:no_activity] precision=0.979 recall=0.939 f1=0.958 support=3298
  [1:plowing    ] precision=0.614 recall=0.287 f1=0.391 support=94
  [2:harvesting ] precision=0.190 recall=0.774 f1=0.305 support=62
  [3:burning    ] precision=0.000 recall=0.000 f1=0.000 support=6
  confusion matrix (rows=true, cols=pred):
[[3096   15  187    0]
 [  49   27   18    0]
 [  12    2   48    0]
 [   6    0    0    0]]


...

TRAIN epoch 7/10:   0%|          | 0/6611 [00:00<?, ?it/s]


=== Epoch 7 complete | TRAIN avg_loss=0.0080 avg_acc=0.9958 | lr now 4.00e-05


VAL inference:   0%|          | 0/433 [00:00<?, ?it/s]

=== FULL VAL REPORT ===
overall_accuracy=0.934  macro_f1=0.365  N=3460
  [0:no_activity] precision=0.967 recall=0.968 f1=0.968 support=3298
  [1:plowing    ] precision=0.567 recall=0.181 f1=0.274 support=94
  [2:harvesting ] precision=0.163 recall=0.339 f1=0.220 support=62
  [3:burning    ] precision=0.000 recall=0.000 f1=0.000 support=6
  confusion matrix (rows=true, cols=pred):
[[3193    6   99    0]
 [  68   17    9    0]
 [  34    7   21    0]
 [   6    0    0    0]]


TRAIN epoch 8/10:   0%|          | 0/6611 [00:00<?, ?it/s]

In [ ]:
# Visualization: metrics (confusion matrix + per-class bars) and
# 3 random patches per class -> end-date preview, true label, predicted label
# ----------------------------
import matplotlib.pyplot as plt

device_for_viz = "cuda" if torch.cuda.is_available() else "cpu"


def plot_confusion_matrix(cm: np.ndarray, save_path: str = "confusion_matrix.png"):
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(ACTIVITY_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(ACTIVITY_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion matrix")
    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved confusion matrix to {save_path}")


def plot_per_class_metrics(report: dict, save_path: str = "per_class_metrics.png"):
    precisions = [report["per_class"][c]["precision"] for c in range(NUM_CLASSES)]
    recalls    = [report["per_class"][c]["recall"]    for c in range(NUM_CLASSES)]
    f1s        = [report["per_class"][c]["f1"]        for c in range(NUM_CLASSES)]

    x = np.arange(NUM_CLASSES)
    w = 0.25
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - w, precisions, width=w, label="precision")
    ax.bar(x,     recalls,    width=w, label="recall")
    ax.bar(x + w, f1s,        width=w, label="f1")
    ax.set_xticks(x); ax.set_xticklabels(ACTIVITY_NAMES)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Per-class metrics (overall_acc={report['overall_accuracy']:.3f}, "
                 f"macro_f1={report['macro_f1']:.3f})")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved per-class metrics chart to {save_path}")


final_report = full_validation_report(cnn, val_loader, device=device_for_viz, num_classes=NUM_CLASSES, use_amp=True)
print_report(final_report)
plot_confusion_matrix(final_report["confusion_matrix"], save_path="confusion_matrix.png")
plot_per_class_metrics(final_report, save_path="per_class_metrics.png")

# SWIR false-color composite, R=SWIR(b5), G=NIR(b4), B=Red(b1).
# Within the 5-band end-date group (0-indexed): b1->0, b4->3, b5->4.
RGB_BAND_IDXS_WITHIN_GROUP = (4, 3, 0)  # (R=b5 SWIR, G=b4 NIR, B=b1 Red)

# Fixed reflectance clip range for display stretch
DISPLAY_MIN = 0.02
DISPLAY_MAX = 0.3

# Bands are 1-indexed b1..b10: end date = b1..b5, start date = b6..b10.
END_DATE_BAND_SLICE = slice(0, 5)
START_DATE_BAND_SLICE = slice(5, 10)

def _load_raw_rgb(path, band_slice, rgb_idxs=RGB_BAND_IDXS_WITHIN_GROUP, vmin=DISPLAY_MIN, vmax=DISPLAY_MAX):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    rgb = arr[band_slice][list(rgb_idxs)]
    rgb = np.transpose(rgb, (1, 2, 0))
    rgb = np.nan_to_num(rgb, nan=0.0, posinf=0.0, neginf=0.0)
    out = np.clip(rgb, vmin, vmax)
    return (out - vmin) / max(1e-6, (vmax - vmin))

def load_raw_start_date_rgb(path):
    return _load_raw_rgb(path, START_DATE_BAND_SLICE)

def load_raw_end_date_rgb(path):
    return _load_raw_rgb(path, END_DATE_BAND_SLICE)


@torch.no_grad()
def predict_single(model, samples, global_idx, device, use_amp=True):
    tmp_ds = TifDatasetLikeTraining(samples, [global_idx], normalize=True)
    x, y, vm = tmp_ds[0]
    x = x.unsqueeze(0).to(device)
    vm = vm.unsqueeze(0).to(device)
    model.eval()
    with torch.amp.autocast("cuda", enabled=(use_amp and device == "cuda")):
        logits = model(x, valid_mask=vm)
    return int(logits.argmax(dim=1).item())


def show_predictions(model, samples, indices, device, classes=(1, 2, 3), n_per_class=3, seed=42):
    rng = random.Random(seed)
    rows = []
    for c in classes:
        pool = [i for i in indices if samples[i].cl == c]
        rows += rng.sample(pool, min(n_per_class, len(pool)))

    fig, axes = plt.subplots(len(rows), 2, figsize=(8, 4 * len(rows)))
    for row, gi in enumerate(rows):
        meta = samples[gi]
        pred = predict_single(model, samples, gi, device)
        color = "green" if pred == meta.cl else "red"

        axes[row, 0].imshow(load_raw_start_date_rgb(meta.path))
        axes[row, 0].axis("off")
        axes[row, 0].set_title("pre-date")

        axes[row, 1].imshow(load_raw_end_date_rgb(meta.path))
        axes[row, 1].axis("off")
        axes[row, 1].set_title(f"target: {ACTIVITY_NAMES[meta.cl]} | pred: {ACTIVITY_NAMES[pred]}", color=color)

    plt.tight_layout()
    plt.show()


show_predictions(cnn, samples, val_idx, device=device_for_viz)